# Benchmark Analysis

## Overview

This notebook documents benchmark methodology, baseline comparisons, and statistical analysis of RL algorithm performance on delayed reward tasks.

---

## Benchmark Design

We evaluate algorithms across three dimensions:
1. **Grid size**: Controls state space complexity (4×4, 8×8, 16×16)
2. **Reward density**: Controls sparsity (3%, 5%, 10% coin coverage)
3. **Random seeds**: Controls statistical confidence (5 seeds per config)

### Evaluation Protocol
- 200 evaluation episodes per configuration
- Monte Carlo return estimation with bootstrapped 95% CIs
- Mann-Whitney U test for pairwise significance testing

## Algorithms Compared

| Algorithm | Type | Delayed Reward Handling |
|-----------|------|------------------------|
| Random Walk | Baseline | None |
| Q-Learning (tabular) | Value-based | TD propagation |
| **Node2Vec + InferNet** | **Embedding-based** | **Auxiliary prediction** |
| DQN | Deep value-based | Neural TD |
| PPO | Policy gradient | GAE |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

# Simulated benchmark results (replace with actual benchmark outputs)
results = {
    'Random': {'g4_d0.10': 0.82, 'g8_d0.05': 0.31, 'g8_d0.03': 0.18, 'g16_d0.03': 0.08},
    'Q-Learning': {'g4_d0.10': 2.41, 'g8_d0.05': 1.12, 'g8_d0.03': 0.67, 'g16_d0.03': 0.29},
    'Node2Vec+InferNet': {'g4_d0.10': 3.85, 'g8_d0.05': 2.74, 'g8_d0.03': 1.93, 'g16_d0.03': 0.94},
    'DQN': {'g4_d0.10': 3.62, 'g8_d0.05': 2.51, 'g8_d0.03': 1.77, 'g16_d0.03': 0.81},
}

configs = list(list(results.values())[0].keys())
x = np.arange(len(configs))
width = 0.2
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (algo, vals) in enumerate(results.items()):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, [vals[c] for c in configs], width, label=algo, color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(['4×4 Dense', '8×8 Medium', '8×8 Sparse', '16×16 Sparse'], fontsize=11)
ax.set_ylabel('Mean Episode Return', fontsize=12)
ax.set_title('Algorithm Comparison Across Benchmark Configurations', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Statistical Analysis

### Improvement Over Random Baseline

In [ ]:
# Compute improvement over random
print('\nImprovement over Random Baseline:')
print(f'{"Algorithm":<25} {"Mean Improvement":>15}')
print('-' * 42)
for algo, vals in results.items():
    if algo == 'Random': continue
    improvement = np.mean([
        vals[c] - results['Random'][c] for c in configs
    ])
    print(f'{algo:<25} {improvement:>+15.2f}')

## Scalability Analysis

How does performance degrade as state space grows?

In [ ]:
# State space vs performance
state_spaces = [16, 64, 256]
node2vec_perf = [3.85, 1.93, 0.94]
dqn_perf = [3.62, 1.77, 0.81]
random_perf = [0.82, 0.18, 0.08]

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogx(state_spaces, node2vec_perf, 'o-', label='Node2Vec+InferNet', color='#2ecc71', lw=2, ms=9)
ax.semilogx(state_spaces, dqn_perf, 's-', label='DQN', color='#f39c12', lw=2, ms=9)
ax.semilogx(state_spaces, random_perf, '^--', label='Random', color='#e74c3c', lw=2, ms=9, alpha=0.7)

ax.set_xlabel('State Space Size (nodes)', fontsize=12)
ax.set_ylabel('Mean Episode Return', fontsize=12)
ax.set_title('Performance vs State Space Complexity', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()